In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler,OneHotEncoder
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression

In [2]:
df=pd.read_csv('Real_estate_featureSelection_v2.csv')

In [3]:
df.head()

,F/H,sector,price,total_area(sqft),bedRoom,bathroom,balcony,agePossession,furnishing_type,additional_room_count,luxury_cat
0,1,81,0.82,949,2,2,3,New,0,0.0,Mid
1,0,50,11.95,3240,3,3,2,Relatively Old,0,2.0,Mid
2,1,107,0.28,493,2,2,2,New,0,0.0,Low
3,0,5,3.50,2367,5,4,2,Old,0,2.0,Low
4,0,3,0.40,450,7,4,3,Relatively New,0,1.0,Low


#### For Regression Models we have to do One hOT Encoding as it gives weightage to feature in  ordinal encoding
#### we can do ordinal encoding for gb and rf(tree based meodels as they didnt see order/mag they make patterns by cut) but for regression ohe needed

In [4]:
### For Regression models we have to do
# 1.OHE
#2.Scaling
#3.log transformation (to normalise right skewed features)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3774 entries, 0 to 3773
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   F/H                    3774 non-null   int64  
 1   sector                 3774 non-null   object 
 2   price                  3774 non-null   float64
 3   total_area(sqft)       3774 non-null   int64  
 4   bedRoom                3774 non-null   int64  
 5   bathroom               3774 non-null   int64  
 6   balcony                3774 non-null   int64  
 7   agePossession          3774 non-null   object 
 8   furnishing_type        3774 non-null   int64  
 9   additional_room_count  3774 non-null   float64
 10  luxury_cat             3774 non-null   object 
dtypes: float64(2), int64(6), object(3)
memory usage: 324.5+ KB


In [6]:

df['sector'].value_counts()

sector
35     174
3      113
102    113
85     110
92     104
      ... 
17A      1
42       1
21A      1
37B      1
17C      1
Name: count, Length: 120, dtype: int64

In [7]:
df['sector'] = df['sector'].astype('str')


In [8]:
df['F/H'] = df['F/H'].replace({
    0: "house",
    1: "flat"
    })

In [9]:
df['furnishing_type'] = df['furnishing_type'].replace({
    0.0: "unfurnished",
    1.0: "semifurnished",
    2.0: "furnished"
})

In [10]:
df=df.rename(columns={'total_area(sqft)':'total_area_sqft'})

In [11]:
df.sample()

,F/H,sector,price,total_area_sqft,bedRoom,bathroom,balcony,agePossession,furnishing_type,additional_room_count,luxury_cat
908,flat,113,1.5,1276,2,2,2,Relatively New,unfurnished,0.0,Mid


In [12]:
x=df.drop(columns=['price'])
y=df['price']

In [13]:
print(x.isna().sum())

F/H                      0
sector                   0
total_area_sqft          0
bedRoom                  0
bathroom                 0
balcony                  0
agePossession            0
furnishing_type          0
additional_room_count    0
luxury_cat               0
dtype: int64


In [14]:
x.shape

(3774, 10)

In [15]:
y.shape

(3774,)

In [16]:
## y_transformation
y_transformed=np.log1p(y)

In [17]:
y_transformed

0       0.598837
1       2.561096
2       0.246860
3       1.504077
4       0.336472
          ...   
3769    1.909543
3770    2.442347
3771    0.470004
3772    0.788457
3773    1.305626
Name: price, Length: 3774, dtype: float64

### ORDINAL ENCODING

In [18]:
encoder_cols=['F/H','sector','agePossession','furnishing_type','luxury_cat']

In [19]:
preprocessor=ColumnTransformer(transformers=[('num',StandardScaler(),['bedRoom','bathroom','total_area_sqft','additional_room_count','balcony']),
                                             ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),encoder_cols)],
                               remainder='passthrough')

In [20]:
pipeline=Pipeline([('preprocessor',preprocessor),
                   ('regressor',LinearRegression())
                  ])

In [21]:
## K-Fold
def apply_cross_validation(pipeline,x,y,scoring='r2'):
    kfold=KFold(n_splits=15,shuffle=True,random_state=42)
    scores=cross_val_score(pipeline,x,y,cv=kfold,scoring=scoring)
    return scores


In [22]:
scores=apply_cross_validation(pipeline,x,y_transformed)


In [23]:
(scores.mean(), scores.std())

(0.5888865676940128, 0.10437499737235914)

In [24]:
def get_mean_absolute_error(model, x, y, test_size=0.3, random_state=42):
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_size, random_state=random_state)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    return mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))

In [25]:
get_mean_absolute_error(pipeline, x, y_transformed)

2.790121655714371

In [26]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [27]:
from sklearn.linear_model import Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor, AdaBoostRegressor
import xgboost
from xgboost import XGBRegressor

models = {
    "linear_reg": LinearRegression(),
    "svr": SVR(),
    "ridge": Ridge(),
    "lasso": Lasso(),
    "decision_tree": DecisionTreeRegressor(),
    "random_forest": RandomForestRegressor(),
    "gradient_boosting": GradientBoostingRegressor(),
    "adaboost": AdaBoostRegressor(),
    "xgboost": XGBRegressor(),
}

In [28]:
def scorer(model_name, pipeline):
    output = [model_name]
    
    scores = apply_cross_validation(pipeline, x, y_transformed)    
    output.append(scores.mean())
    
    output.append(get_mean_absolute_error(pipeline, x, y_transformed))
    
    return output

In [29]:
preprocessor=ColumnTransformer(transformers=[('num',StandardScaler(),['bedRoom','bathroom','balcony','total_area_sqft','additional_room_count']),
                                             ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),encoder_cols)]
                               
                               ,remainder='passthrough')

model_output=[]
for model_name,model in models.items():
    pipeline=Pipeline([('preprocessor',preprocessor),
                   ('regressor',model)
                  ])
    model_output.append(scorer(model_name,pipeline))

In [30]:
model_output

[['linear_reg', 0.5888865676940126, 2.7901216557143935],
 ['svr', 0.6291726721793949, 1.4748894722577799],
 ['ridge', 0.5888949955191713, 2.789661732122857],
 ['lasso', 0.014145112112315943, 1.5494546437621242],
 ['decision_tree', 0.7976412285255098, 0.6371596645857188],
 ['random_forest', 0.8826014932017603, 0.5415163421950292],
 ['gradient_boosting', 0.8645267496109624, 0.6104566570351763],
 ['adaboost', 0.7263496960810716, 0.9341771910231793],
 ['xgboost', 0.8873018362004654, 0.5233954334764068]]

In [31]:
model_df=pd.DataFrame(model_output,columns=['model','r2','mae'])
model_df.sort_values(['mae'])

,model,r2,mae
8,xgboost,0.887302,0.523395
5,random_forest,0.882601,0.541516
6,gradient_boosting,0.864527,0.610457
4,decision_tree,0.797641,0.637160
7,adaboost,0.726350,0.934177
1,svr,0.629173,1.474889
3,lasso,0.014145,1.549455
2,ridge,0.588895,2.789662
0,linear_reg,0.588887,2.790122


### OHE 

In [32]:
preprocessor=ColumnTransformer(transformers=[('num',StandardScaler(),['bedRoom','bathroom','total_area_sqft','balcony','additional_room_count']),
                                             ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),encoder_cols),
                                             ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','furnishing_type'])]
                               ,remainder='passthrough')

In [33]:
x.columns

Index(['F/H', 'sector', 'total_area_sqft', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'furnishing_type', 'additional_room_count',
       'luxury_cat'],
      dtype='object')

In [34]:
pipeline=Pipeline([('preprocessor',preprocessor),
                  ('regressor',LinearRegression())
                  ] )

In [35]:
kfold=KFold(n_splits=10,shuffle=True,random_state=42)
scores=cross_val_score(pipeline,x,y_transformed,cv=kfold,scoring='r2',error_score='raise')

D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: F

In [36]:
(scores.mean(),scores.std())

(0.7551398435800009, 0.09348552365056341)

In [37]:
get_mean_absolute_error(pipeline, x, y_transformed)

D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


0.9831976550045307

In [38]:
preprocessor=ColumnTransformer(transformers=[('num',StandardScaler(),['bedRoom','bathroom','total_area_sqft','balcony','additional_room_count']),
                                             ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),encoder_cols),
                                             ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','furnishing_type'])]
                               ,remainder='passthrough')

model_output_ohe=[]
for model_name,model in models.items():
    pipeline=Pipeline([('preprocessor',preprocessor),
                  ('regressor',model)
                  ] )
    model_output_ohe.append(scorer(model_name, pipeline))

D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
D:\anaconda\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: F

In [39]:
model_output_ohe

[['linear_reg', 0.7666846272517415, 0.9831976550045307],
 ['svr', 0.6360991371210022, 1.4385916913830956],
 ['ridge', 0.7663479696624936, 1.024262929396485],
 ['lasso', 0.01414511211231561, 1.5494546437621226],
 ['decision_tree', 0.8127714532698483, 0.6267299165661123],
 ['random_forest', 0.8915455540863954, 0.5117621641050069],
 ['gradient_boosting', 0.8665621760705411, 0.6202959682781959],
 ['adaboost', 0.7257936952296002, 0.9340505421227081],
 ['xgboost', 0.8924476076327219, 0.5345901553313077]]

In [40]:
model_df_ohe=pd.DataFrame(model_output_ohe,columns=['model_name','r2','mae'])
model_df_ohe.sort_values('mae')

,model_name,r2,mae
5,random_forest,0.891546,0.511762
8,xgboost,0.892448,0.534590
6,gradient_boosting,0.866562,0.620296
4,decision_tree,0.812771,0.626730
7,adaboost,0.725794,0.934051
0,linear_reg,0.766685,0.983198
2,ridge,0.766348,1.024263
1,svr,0.636099,1.438592
3,lasso,0.014145,1.549455


#### if there were so much rows then we need to do PCA to reduce dimensionality as ohe increses features but for 3000 rows without pca also model gives good results

### Target Encoder

In [41]:
pip install category_encoders

Note: you may need to restart the kernel to use updated packages.


In [42]:
import category_encoders as ce

preprocessor=ColumnTransformer(transformers=[('num',StandardScaler(),['bedRoom','bathroom','total_area_sqft','balcony','additional_room_count']),
                                             ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),encoder_cols),
                                             ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['agePossession','furnishing_type']),
                                            ('targetencoder',ce.TargetEncoder(),['sector'])]
                               ,remainder='passthrough')

model_output_te=[]
for model_name,model in models.items():
    pipeline=Pipeline([('preprocessor',preprocessor),
                  ('regressor',model)
                  ] )
    model_output_te.append(scorer(model_name, pipeline))

In [43]:
model_df_te=pd.DataFrame(model_output_te,columns=['model_name','r2','mae'])
model_df_te.sort_values('mae')

,model_name,r2,mae
5,random_forest,0.898483,0.501519
8,xgboost,0.901420,0.522999
6,gradient_boosting,0.880855,0.573574
4,decision_tree,0.798352,0.736505
7,adaboost,0.786513,0.844076
0,linear_reg,0.734921,1.181010
2,ridge,0.734922,1.184325
1,svr,0.664890,1.393572
3,lasso,0.014145,1.549455


### Hyperparameter Tuning

In [45]:
from sklearn.model_selection import GridSearchCV

In [57]:
param_grid = {
    'regressor__n_estimators': [100, 200,300,500],
    'regressor__max_depth': [None,3, 5, 7],
    'regressor__learning_rate': [0.01,0.05, 0.1]
}

In [58]:
preprocessor=ColumnTransformer(transformers=[('num',StandardScaler(),['bedRoom','bathroom','total_area_sqft','balcony','additional_room_count']),
                                             ('cat',OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),encoder_cols),
                                             ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['agePossession','furnishing_type']),
                                            ('targetencoder',ce.TargetEncoder(),['sector'])]
                               ,remainder='passthrough')

pipeline=Pipeline([('preprocessor',preprocessor),
                   ('regressor',XGBRegressor())
                  ])

In [59]:

kfold=KFold(n_splits=10,shuffle=True,random_state=42)
grid=GridSearchCV(pipeline,param_grid,cv=kfold,scoring='r2',n_jobs=-1,verbose=2)

In [ ]:
VERBOSE:
Controls how much information is printed during training.
Values:
verbose = 0 → no output
verbose = 1 → basic info
verbose = 2 → fold-wise progress (recommended )
verbose = 3 → more detailed
verbose = 4 → full logs (start/end of each fit

In [60]:
grid.fit(x,y_transformed)

Fitting 10 folds for each of 48 candidates, totalling 480 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__learning_rate': [0.01, 0.05, ...], 'regressor__max_depth': [None, 3, ...], 'regressor__n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the compu

In [61]:
grid.best_params_

{'regressor__learning_rate': 0.1,
 'regressor__max_depth': 5,
 'regressor__n_estimators': 500}

In [62]:
grid.best_score_

0.9007918657971082

### Trying out Accuracy By testing 

In [69]:
x.columns

Index(['F/H', 'sector', 'total_area_sqft', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'furnishing_type', 'additional_room_count',
       'luxury_cat'],
      dtype='object')

In [70]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3774 entries, 0 to 3773
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   F/H                    3774 non-null   object 
 1   sector                 3774 non-null   object 
 2   total_area_sqft        3774 non-null   int64  
 3   bedRoom                3774 non-null   int64  
 4   bathroom               3774 non-null   int64  
 5   balcony                3774 non-null   int64  
 6   agePossession          3774 non-null   object 
 7   furnishing_type        3774 non-null   object 
 8   additional_room_count  3774 non-null   float64
 9   luxury_cat             3774 non-null   object 
dtypes: float64(1), int64(4), object(5)
memory usage: 295.0+ KB


In [76]:
df.iloc[5]

F/H                                flat
sector                              113
price                              3.72
total_area_sqft                    2660
bedRoom                               5
bathroom                              4
balcony                               3
agePossession            Relatively New
furnishing_type           semifurnished
additional_room_count               1.0
luxury_cat                          Low
Name: 5, dtype: object

In [77]:
data = [['flat', '113',2660, 5, 4, 3, 'Relatively New', 'semifurnished','1.0','Low',]]
columns = ['F/H', 'sector', 'total_area_sqft', 'bedRoom', 'bathroom',
       'balcony', 'agePossession', 'furnishing_type', 'additional_room_count',
       'luxury_cat']
       

In [78]:
test_df=pd.DataFrame(data,columns=columns)
test_df

,F/H,sector,total_area_sqft,bedRoom,bathroom,balcony,agePossession,furnishing_type,additional_room_count,luxury_cat
0,flat,113,2660,5,4,3,Relatively New,semifurnished,1.0,Low


In [79]:
np.expm1(grid.best_estimator_.predict(test_df))

array([3.76012], dtype=float32)

### Export

In [81]:
import pickle
with open('Real_estate_pipeline.pkl', 'wb') as file:
    pickle.dump(grid.best_estimator_, file)
with open('Real_estate_df.pkl', 'wb') as file:
    pickle.dump(x, file)

In [83]:
get_mean_absolute_error(grid.best_estimator_, x, y_transformed)

0.5307667137423344